# Análisis derivados

Analiza las reseñas sin representación léxica, la coherencia con corpus de referencia de tamaño común y la robustez de los resultados. Las celdas 1 a 6 comparten variables y deben ejecutarse en orden.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Reseñas sin representación léxica (Sección 4.3)

In [ ]:
# ============================================================
# M1 — Fracción de documentos sin contenido tras el preprocesamiento
# NO reentrena. Re-vectoriza (determinista) + lee los df_*_final ya guardados.
# ============================================================
import pickle, numpy as np, pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"

d = pickle.load(open(f"{DFDIR}/prep_lemmatizado.pkl", "rb"))
corpus, final_stopwords = d['corpus'], d['final_stopwords']
df_lda   = pd.read_pickle(f"{DFDIR}/df_lda_final.pkl")
df_tfidf = pd.read_pickle(f"{DFDIR}/df_tfidf_final.pkl")

assert len(corpus) == len(df_lda) == len(df_tfidf), \
    f"DESALINEADO corpus={len(corpus)} lda={len(df_lda)} tfidf={len(df_tfidf)}"
lc = np.array([len(c.split()) for c in corpus])
lt = np.array([len(str(t).split()) for t in df_lda['text']])
print(f"corpus={len(corpus):,} docs | stopwords={len(final_stopwords)}")
print(f"CHEQUEO alineación corpus<->df_lda: corr={np.corrcoef(lc, lt)[0,1]:.3f}  (debe ser > 0.8)")
print(f"CHEQUEO ids df_lda == df_tfidf: {bool((df_lda['id'].to_numpy()==df_tfidf['id'].to_numpy()).all())}")

# ---------- M1.1  filas vacías en la matriz documento-término ----------
cv = CountVectorizer(stop_words=final_stopwords, min_df=10, max_df=0.6, ngram_range=(1,2))
X  = cv.fit_transform(corpus)
tv = TfidfVectorizer(stop_words=final_stopwords, min_df=10, max_df=0.6,
                     ngram_range=(1,2), sublinear_tf=True)
Xt = tv.fit_transform(corpus)
vac    = np.asarray(X.getnnz(axis=1) == 0)
vac_tf = np.asarray(Xt.getnnz(axis=1) == 0)
print(f"\n### M1.1")
print(f"X  LDA   {X.shape}  vacías = {vac.sum():,}  ({vac.mean():.2%})")
print(f"X  TFIDF {Xt.shape}  vacías = {vac_tf.sum():,}  ({vac_tf.mean():.2%})")
print(f"¿mismo conjunto de filas vacías?  {bool((vac == vac_tf).all())}")

# ---------- M1.2  verificación empírica del mecanismo de LDA ----------
probs = df_lda[[f'prob_topico_{i}' for i in range(6)]].to_numpy()
tema  = df_lda['tema_dominante'].to_numpy()
print(f"\n### M1.2")
if vac.sum() == 0:
    print("NO HAY FILAS VACÍAS -> la compuerta se cierra acá.")
else:
    sub   = probs[vac]
    rango = sub.max(axis=1) - sub.min(axis=1)
    print(f"filas vacías: {vac.sum():,}")
    print(f"(a) rango máx-mín dentro de la fila: max={rango.max():.3e}  media={rango.mean():.3e}")
    print(f"    seis probabilidades iguales (tol 1e-9) en {(rango < 1e-9).mean():.2%} de las filas")
    print(f"    ejemplo de fila vacía: {np.round(sub[0], 6)}   (1/6 = {1/6:.6f})")
    print(f"(b) argmax == 0 en {(tema[vac] == 0).mean():.2%} de las filas vacías")
    print(f"    reparto de argmax en filas vacías: {sorted(Counter(tema[vac]).items())}")
    print(f"    control — rango medio en filas NO vacías: {(probs[~vac].max(1)-probs[~vac].min(1)).mean():.3f}")

# ---------- M1.3  composición del T0 de LDA  <<< LA COMPUERTA ----------
T0 = (tema == 0)
print(f"\n### M1.3   *** COMPUERTA ***")
print(f"T0 = {T0.sum():,} docs ({T0.mean():.2%} del corpus)")
if T0.sum():
    print(f">>> vacías dentro de T0: {(T0 & vac).sum():,}  =  {(T0 & vac).sum()/T0.sum():.2%} de T0")
print(f"T0 depurado de vacías: {(T0 & ~vac).sum():,} "
      f"({(T0 & ~vac).sum()/max((~vac).sum(),1):.2%} del corpus con contenido)")
print("\nDistribución de tópicos LDA — tal cual vs. excluyendo vacías (%):")
comp = pd.DataFrame({
    'todos':     pd.Series(tema).value_counts(normalize=True).sort_index()*100,
    'sin_vacias':pd.Series(tema[~vac]).value_counts(normalize=True).sort_index()*100,
}).round(1)
print(comp.to_string())

# ---------- M1.4  cluster residual de TF-IDF ----------
cl  = df_tfidf['cluster_tema'].to_numpy()
tam = pd.Series(cl).value_counts().sort_index()
res = int(tam.idxmax()); enR = (cl == res)
print(f"\n### M1.4")
print((tam/len(cl)*100).round(1).to_string())
print(f"cluster residual = {res}  ({tam[res]:,} docs, {tam[res]/len(cl):.1%})")
print(f"vectores nulos dentro del residual: {(enR & vac).sum():,}  = {(enR & vac).sum()/enR.sum():.2%} del cluster")
print(f"¿TODOS los nulos caen en ese cluster? {bool((cl[vac] == res).all())}")
print(f"clusters que reciben nulos: {sorted(Counter(cl[vac]).items())}")

# ---------- M1.5  por parque ----------
tab = pd.DataFrame({'parque': df_lda['nombre_parque'].to_numpy(), 'vacia': vac, 'T0': T0})
r = tab.groupby('parque').agg(n=('vacia','size'), pct_vacias=('vacia','mean'), pct_T0=('T0','mean'))
r[['pct_vacias','pct_T0']] *= 100
r['T0_entre_no_vacias'] = tab[~tab.vacia].groupby('parque')['T0'].mean()*100
r = r.round(1).sort_values('pct_T0', ascending=False)
print(f"\n### M1.5\n{r.to_string()}")
print(f"\nPearson  %vacías vs %T0 : {r.pct_vacias.corr(r.pct_T0):.3f}")
print(f"Spearman %vacías vs %T0 : {r.pct_vacias.corr(r.pct_T0, method='spearman'):.3f}   (n=10 parques)")

# ---------- M1.6  descomponer el 27 % de W2V (sin entrenar) ----------
w = pickle.load(open(f"{DFDIR}/prep_w2v_tokens.pkl", "rb")); toks = w['tokens']
frq = Counter(t for doc in toks for t in doc)
vocab = {t for t, c in frq.items() if c >= 5}          # min_count=5, idéntico al de gensim
n = len(toks)
sin_tok = sum(1 for dd in toks if len(dd) == 0)
sin_voc = sum(1 for dd in toks if dd and not any(t in vocab for t in dd))
print(f"\n### M1.6")
print(f"vocabulario reconstruido min_count=5: {len(vocab):,}   (el documento dice 2.846)")
print(f"docs        : {n:,}")
print(f"descartados : {sin_tok+sin_voc:,} ({(sin_tok+sin_voc)/n:.2%})   (el documento dice ~27 %)")
print(f"  · por el DICCIONARIO (0 tokens tras filtrar): {sin_tok:,} ({sin_tok/n:.2%})")
print(f"  · por min_count=5 (tiene tokens, ninguno en vocab): {sin_voc:,} ({sin_voc/n:.2%})")
print(f"válidos     : {n-sin_tok-sin_voc:,}   (el notebook guardó 19.925)")

# ---------- M1.7  de qué están hechas las reseñas vacías ----------
txt = df_lda['text'].to_numpy(); idx = np.where(vac)[0]
print(f"\n### M1.7   30 reseñas al azar entre las vacías")
if len(idx):
    for i in np.random.default_rng(42).choice(idx, size=min(30, len(idx)), replace=False):
        print(f"  [{len(str(txt[i]).split()):>2} pal] {str(txt[i])[:110]}")
    L = pd.Series([len(str(t).split()) for t in txt])
    print(f"\nlongitud original (palabras) — vacías: mediana={L[vac].median():.0f} media={L[vac].mean():.1f}"
          f" | no vacías: mediana={L[~vac].median():.0f} media={L[~vac].mean():.1f}")

# ---------- M1.8  ¿es el mismo conjunto en las cinco arquitecturas? ----------
print(f"\n### M1.8   solapamiento entre mecanismos")
ids_vac = set(df_lda.loc[vac, 'id'])
base = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")
tot = len(base); p_vac = len(ids_vac)/tot

db = pd.read_pickle(f"{DFDIR}/df_bertopic_final.pkl")
colb = next(c for c in db.columns if 'topic' in c.lower() or 'topico' in c.lower())
ids_ruido = set(db.loc[db[colb] == -1, 'id'])
inter = len(ids_vac & ids_ruido)
print(f"BERTopic ruido={len(ids_ruido):,} ({len(ids_ruido)/tot:.1%}) | vacías={len(ids_vac):,}")
print(f"  intersección={inter:,}  observado={inter/max(len(ids_ruido),1):.1%} del ruido"
      f"  vs esperado por azar={p_vac:.1%}   lift={inter/max(len(ids_ruido)*p_vac,1e-9):.2f}")

ids_w2v_ok = set(pd.read_pickle(f"{DFDIR}/df_w2v_final.pkl")['id'])
ids_desc = set(base['id']) - ids_w2v_ok
i2 = len(ids_vac & ids_desc)
print(f"W2V descartadas={len(ids_desc):,} ({len(ids_desc)/tot:.1%})")
print(f"  intersección={i2:,}  observado={i2/max(len(ids_desc),1):.1%} del descarte"
      f"  vs esperado={p_vac:.1%}   lift={i2/max(len(ids_desc)*p_vac,1e-9):.2f}")

llm = pd.read_csv(f"{BD}/Topic/Resultados/Topico/LLM_results_with_id.csv")
print(f"LLM: distribución de topic_id sobre {len(llm):,} reseñas (%):")
print((llm['topic_id'].value_counts(normalize=True).sort_index()*100).round(1).to_string())
llm['vacia'] = llm['id'].isin(ids_vac)
print("  % de vacías dentro de cada topic_id del LLM:")
print((llm.groupby('topic_id')['vacia'].mean()*100).round(1).to_string())

### Distribución por parque con y sin esas reseñas (Tabla 4.5)

In [ ]:
perfil_todos = pd.crosstab(tab.parque, pd.Series(tema, index=tab.index), normalize='index')*100
perfil_limpio = pd.crosstab(tab.parque[~vac], pd.Series(tema, index=tab.index)[~vac], normalize='index')*100
print("=== A: tal cual (=Figura 4.2) ===\n", perfil_todos.round(1).to_string())
print("\n=== B: excluyendo las 7.404 vacías ===\n", perfil_limpio.round(1).to_string())
print("\ntópico modal por parque —  A:", perfil_todos.idxmax(axis=1).to_dict())
print("tópico modal por parque —  B:", perfil_limpio.idxmax(axis=1).to_dict())
print("\nn por parque tras excluir vacías:\n", (~tab.vac if False else tab[~tab.vacia].groupby('parque').size()).to_string())

### Distribución por técnica (Tabla 4.3), palabras del LLM (Figura 4.2) y nombres de parque (Sección 4.2.1)

In [ ]:
# ============================================================
# M2 (distribución por arquitectura, A y B) + M8 (top-10 del LLM) + M9 (nombres de parque)
# Continúa la sesión de M1: reutiliza vac, df_lda, ids_vac, DFDIR, BD
# ============================================================
import pickle, numpy as np, pandas as pd

base = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")
ids_vac = set(df_lda.loc[vac, 'id'])

def AB(ids, lab, nombre):
    s = pd.DataFrame({'id': ids, 'lab': lab})
    a = s['lab'].value_counts(normalize=True).sort_index()*100
    b = s.loc[~s['id'].isin(ids_vac), 'lab'].value_counts(normalize=True).sort_index()*100
    t = pd.DataFrame({'A_tal_cual': a.round(1), 'B_sin_vacias': b.round(1)})
    t.index.name = f"{nombre}  (n_A={len(s):,}  n_B={(~s['id'].isin(ids_vac)).sum():,})"
    print(f"\n{t.to_string()}")

print("### M2 — % de documentos por tópico")
AB(df_lda['id'].to_numpy(), df_lda['tema_dominante'].to_numpy(), 'LDA')
dt = pd.read_pickle(f"{DFDIR}/df_tfidf_final.pkl");  AB(dt['id'].to_numpy(), dt['cluster_tema'].to_numpy(), 'K-Means + TF-IDF')
dw = pd.read_pickle(f"{DFDIR}/df_w2v_final.pkl");    AB(dw['id'].to_numpy(), dw['cluster_w2v'].to_numpy(), 'K-Means + Word2Vec')
db = pd.read_pickle(f"{DFDIR}/df_bertopic_final.pkl")
colb = next(c for c in db.columns if 'topic' in c.lower() or 'topico' in c.lower())
AB(db['id'].to_numpy(), db[colb].to_numpy(), f'BERTopic (col={colb}; -1 = ruido)')
dl = pd.read_csv(f"{BD}/Topic/Resultados/Topico/LLM_results_with_id.csv")
AB(dl['id'].to_numpy(), dl['topic_id'].to_numpy(), 'LLM zero-shot')

# ---------- M8 — top-10 del LLM, que 08_metricas calcula y no imprime ----------
import unicodedata, spacy
from sklearn.feature_extraction.text import TfidfVectorizer
print("\n### M8 — top-10 del LLM bajo el diccionario final")
res = pd.read_csv(f"{BD}/Topic/Resultados/Topico/LLM_results.csv")
nlp = spacy.load('es_core_news_sm', disable=['parser','ner'])
res['lem'] = [' '.join(t.lemma_ for t in d if t.is_alpha)
              for d in nlp.pipe(res['text'].astype(str).tolist(), batch_size=200)]
for tid in sorted(res['topic_id'].unique()):
    if tid == -1: continue
    tx = [t for t in res.loc[res.topic_id == tid, 'lem'] if t.strip()]
    if len(tx) < 3: continue
    v = TfidfVectorizer(max_features=200, min_df=2, stop_words=final_stopwords, strip_accents='unicode')
    M = v.fit_transform(tx).toarray(); nm = v.get_feature_names_out()
    print(f"  tóp{tid} (n={len(tx):,}): {[nm[i] for i in M.mean(0).argsort()[-10:][::-1]]}")

# ---------- M9 — nombres de parque en las palabras representativas ----------
print("\n### M9 — tokens de nombres de parque en los top-10 (las 10 semillas)")
def rm(t): return ''.join(c for c in unicodedata.normalize('NFD', str(t)) if unicodedata.category(c) != 'Mn')
NOM = set()
for n in base['nombre_parque'].str.lower().unique():
    for p in rm(n).replace('-', ' ').replace('_', ' ').split():
        if len(p) > 2: NOM.add(p)
NOM |= {'rosedal', 'rodo', 'batlle', 'ordonez', 'vilarino', 'yeguas', 'dolores', 'herrera'}
print(f"tokens buscados ({len(NOM)}): {sorted(NOM)}")
for ck, lab in [("ckpt_lda_multiseed.pkl","LDA"), ("ckpt_tfidf_multiseed.pkl","TF-IDF"),
                ("ckpt_w2v_multiseed.pkl","W2V"), ("ckpt_bertopic_multiseed.pkl","BERTopic")]:
    d = pickle.load(open(f"{DFDIR}/{ck}", "rb")); sw = set(d['final_stopwords']); hits = []
    for s, tw in d['topic_words_raw_by_seed'].items():
        for ti, row in enumerate(tw):
            top10 = [w for w in row if w not in sw][:10]
            for pos, w in enumerate(top10, 1):
                if any(p in w.split() or p == w for p in NOM):
                    hits.append((s, ti, pos, w))
    print(f"\n[{lab}] {len(hits)} apariciones en top-10 sobre 10 semillas x 6 tópicos")
    for s, ti, pos, w in sorted(hits)[:25]:
        print(f"   semilla {s:>2} · tóp{ti} · posición {pos:>2} · '{w}'")

### Destino de las reseñas sin representación (Tabla 4.4)

In [ ]:
print("### conteos exactos — destino del residuo por arquitectura")
for nm, ids, lab in [('LDA', df_lda['id'], df_lda['tema_dominante']),
                     ('TF-IDF', dt['id'], dt['cluster_tema']),
                     ('BERTopic', db['id'], db[colb]),
                     ('LLM', dl['id'], dl['topic_id'])]:
    s = pd.DataFrame({'id': ids.to_numpy(), 'lab': np.asarray(lab)})
    s['vac'] = s['id'].isin(ids_vac)
    t = pd.crosstab(s['lab'], s['vac'])
    t.columns = ['con_contenido', 'vacia']
    t['total'] = t.sum(1)
    t['pct_del_residuo'] = (t['vacia'] / t['vacia'].sum() * 100).round(1)
    t['pct_del_topico_es_residuo'] = (t['vacia'] / t['total'] * 100).round(1)
    print(f"\n[{nm}]  residuo total en esta arquitectura = {t['vacia'].sum():,}\n{t.to_string()}")
print(f"\n[W2V] descartadas que son residuo: 7,094 / 7,395  |  residuo que sobrevive en W2V: 310")

### Reseñas cortas, longitud, Davies-Bouldin, nombres de parque y coherencia del LLM

M4: Sección 3.4.4 · M5: Sección 3.4.3 · M7: Tabla 3.2 · M9b: Sección 4.2.1 · M6: Tabla 4.1, fila LLM.

In [ ]:
# ============================================================
# M4 · M5 · M7 · M9b · M6
# Continúa la sesión: reutiliza corpus, vac, df_lda, final_stopwords, BD, DFDIR, cv
# ============================================================
import pickle, numpy as np, pandas as pd
from collections import Counter

# ---------- M4 · descomponer el "indeterminado" sin volver a correr lingua ----------
base = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")
corto = base['text'].astype(str).str.strip().str.len() < 15
print("### M4")
print(f"reseñas con < 15 caracteres (indeterminado por regla, no por el detector): "
      f"{corto.sum():,} ({corto.mean():.2%})   |  'indeterminado' total reportado: 32,48 %")
print(f"→ el detector realmente falló en ~{32.48 - corto.mean()*100:.2f} puntos del corpus")
ids_vac_s = set(df_lda.loc[vac, 'id'])
b = base.assign(vacia=base['id'].isin(ids_vac_s), corta=corto)
print(pd.crosstab(b.corta, b.vacia, normalize='all').mul(100).round(1).to_string())
print(f"P(corta | vacía) = {b.loc[b.vacia,'corta'].mean():.1%}   "
      f"P(vacía | corta) = {b.loc[b.corta,'vacia'].mean():.1%}")

# ---------- M5 · longitud antes y después del filtrado ----------
uni = {w for w in cv.get_feature_names_out() if ' ' not in w}
brutos    = np.array([len([t for t in str(x).split() if t.isalpha()]) for x in df_lda['text']])
contenido = np.array([sum(1 for t in c.split() if t in uni) for c in corpus])
print("\n### M5")
for nm, a in [('tokens alfabéticos del texto original', brutos), ('términos del vocabulario final', contenido)]:
    q = np.percentile(a, [25, 50, 75])
    print(f"{nm:<40} total={a.sum():,}  media={a.mean():.2f}  mediana={q[1]:.0f}  Q1={q[0]:.0f}  Q3={q[2]:.0f}")
for u in (0, 1, 2, 3):
    print(f"  reseñas con <= {u} términos de contenido: {(contenido <= u).sum():,} ({(contenido <= u).mean():.2%})")
print(f"→ con una media de {contenido.mean():.1f} términos por reseña, la ventana deslizante de diez "
      f"cubre el documento entero en el {(contenido <= 10).mean():.1%} de los casos")

# ---------- M7 · Davies-Bouldin sobre las diez semillas ----------
print("\n### M7")
for f, lab in [("barrido_tfidf_multiseed.pkl", "K-Means + TF-IDF"), ("barrido_w2v_multiseed.pkl", "K-Means + W2V")]:
    d = pickle.load(open(f"{DFDIR}/{f}", "rb")); br = d['barrido']
    ks = d['k_range']
    ch = pd.DataFrame({s: {k: br[s][k][0] for k in ks} for s in br})
    db_ = pd.DataFrame({s: {k: br[s][k][1] for k in ks} for s in br})
    print(f"\n[{lab}]")
    print(pd.DataFrame({'CH_media': ch.mean(1).round(1), 'CH_sd': ch.std(1).round(1),
                        'DB_media': db_.mean(1).round(3), 'DB_sd': db_.std(1).round(3)}).to_string())
    print(f"  K óptimo por CH (argmax) por semilla: {dict(ch.idxmax())}")
    print(f"  K óptimo por DB (argmin) por semilla: {dict(db_.idxmin())}")
d = pickle.load(open(f"{DFDIR}/barrido_lda_multiseed.pkl", "rb"))
cvv = pd.DataFrame(d['barrido'])
print(f"\n[LDA · C_v]\n{pd.DataFrame({'Cv_media': cvv.mean(1).round(4), 'Cv_sd': cvv.std(1).round(4)}).to_string()}")
print(f"  K óptimo por C_v por semilla: {dict(cvv.idxmax())}")

# ---------- M9b · robustez a la fuga de nombres de parque ----------
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
ESTRICTO = {'rosedal', 'rodo', 'batlle', 'ordoniez', 'ordonez', 'vilarinio', 'vilarino',
            'yeguas', 'dolores', 'garcia', 'fernando'}
AMPLIO   = ESTRICTO | {'museo', 'modelo', 'idea', 'villa', 'punta', 'jose'}
print("\n### M9b — Tabla 4.1 recalculada excluyendo tokens de nombre de parque")
for etiqueta, EXTRA in [('Tabla 4.1 (como está)', set()), ('estricto', ESTRICTO), ('amplio', AMPLIO)]:
    fila = {}
    for ck, lab in [("ckpt_lda_multiseed.pkl","LDA"), ("ckpt_tfidf_multiseed.pkl","TF-IDF"),
                    ("ckpt_w2v_multiseed.pkl","W2V"), ("ckpt_bertopic_multiseed.pkl","BERTopic")]:
        dd = pickle.load(open(f"{DFDIR}/{ck}", "rb")); sw = set(dd['final_stopwords']) | EXTRA
        ref = dd['ref_texts']; dic = Dictionary(ref); cs, ts = [], []
        for s, tw in dd['topic_words_raw_by_seed'].items():
            tp = [[w for w in row if w not in sw and not any(p in w.split() for p in EXTRA)][:10] for row in tw]
            per = CoherenceModel(topics=tp, texts=ref, dictionary=dic,
                                 coherence='c_npmi').get_coherence_per_topic()
            cs.append(float(np.nanmean(per)))
            allw = [w for t in tp for w in t]; ts.append(len(set(allw))/len(allw))
        fila[lab] = (np.mean(cs), np.std(cs), np.mean(ts), np.std(ts))
    print(f"\n-- {etiqueta} --")
    for lab, (c, cd, t, td) in fila.items():
        print(f"  {lab:<9} C_npmi = {c:+.3f} ± {cd:.3f}   TD = {t:.3f} ± {td:.3f}")

# ---------- M6 · los tres confundidos de la C_npmi del LLM ----------
import unicodedata, spacy
print("\n### M6")
def rm(t): return ''.join(c for c in unicodedata.normalize('NFD', str(t)) if unicodedata.category(c) != 'Mn')
res = pd.read_csv(f"{BD}/Topic/Resultados/Topico/LLM_results.csv")
nlp = spacy.load('es_core_news_sm', disable=['parser','ner'])
lem = [' '.join(t.lemma_ for t in d if t.is_alpha)
       for d in nlp.pipe(res['text'].astype(str).tolist(), batch_size=200)]
sw = set(final_stopwords)
ref_con = [rm(t).split() for t in lem]                              # como está hoy
ref_sin = [[w for w in rm(t).split() if w not in sw] for t in lem]   # como las otras cuatro

from sklearn.feature_extraction.text import TfidfVectorizer
tw_llm = []
for tid in sorted(res['topic_id'].unique()):
    if tid == -1: continue
    tx = [t for t in pd.Series(lem)[res['topic_id'].values == tid] if t.strip()]
    if len(tx) < 3: continue
    v = TfidfVectorizer(max_features=200, min_df=2, stop_words=final_stopwords, strip_accents='unicode')
    M = v.fit_transform(tx).toarray(); nm_ = v.get_feature_names_out()
    tw_llm.append([nm_[i] for i in M.mean(0).argsort()[-10:][::-1]])

for et, ref in [('(actual) con stopwords', ref_con), ('(a) SIN stopwords', ref_sin)]:
    dic = Dictionary(ref)
    c = CoherenceModel(topics=tw_llm, texts=ref, dictionary=dic, coherence='c_npmi').get_coherence()
    print(f"  LLM {et:<26} C_npmi = {c:+.4f}   (Tabla 4.1 dice -0,206)")

print("\n(b) las otras tres sobre un corpus de referencia de 2.000 — 10 submuestras")
rng = np.random.default_rng(42)
for ck, lab in [("ckpt_lda_multiseed.pkl","LDA"), ("ckpt_tfidf_multiseed.pkl","TF-IDF"),
                ("ckpt_bertopic_multiseed.pkl","BERTopic")]:
    dd = pickle.load(open(f"{DFDIR}/{ck}", "rb")); swd = set(dd['final_stopwords'])
    ref_full = dd['ref_texts']
    tp = [[w for w in row if w not in swd][:10] for row in dd['topic_words_raw_by_seed'][42]]
    vals = []
    for _ in range(10):
        sub = [ref_full[i] for i in rng.choice(len(ref_full), 2000, replace=False)]
        dic = Dictionary(sub)
        per = CoherenceModel(topics=tp, texts=sub, dictionary=dic,
                             coherence='c_npmi').get_coherence_per_topic()
        vals.append(float(np.nanmean(per)))
    print(f"  {lab:<9} sobre 2.000: {np.mean(vals):+.4f} ± {np.std(vals):.4f}   "
          f"| sobre el corpus completo (Tabla 4.1): LDA +0,049 / TF-IDF +0,075 / BERTopic +0,064")

### Longitud del corpus de referencia (Tabla 3.1, Sección 4.2.2)

In [ ]:
# 1) completar el panel de 2.000 con W2V
# 2) ¿importa el factor 1,4 entre las cuatro multi-semilla?  BERTopic 27.320 -> 19.925
for ck, lab, n_sub in [("ckpt_w2v_multiseed.pkl", "W2V", 2000),
                       ("ckpt_bertopic_multiseed.pkl", "BERTopic", 19925)]:
    dd = pickle.load(open(f"{DFDIR}/{ck}", "rb")); swd = set(dd['final_stopwords'])
    ref_full = dd['ref_texts']
    tp = [[w for w in row if w not in swd][:10] for row in dd['topic_words_raw_by_seed'][42]]
    vals = []
    for _ in range(10):
        sub = [ref_full[i] for i in rng.choice(len(ref_full), n_sub, replace=False)]
        per = CoherenceModel(topics=tp, texts=sub, dictionary=Dictionary(sub),
                             coherence='c_npmi').get_coherence_per_topic()
        vals.append(float(np.nanmean(per)))
    print(f"{lab:<9} ref de {n_sub:,}: {np.mean(vals):+.4f} ± {np.std(vals):.4f}")

# 3) longitud del corpus de referencia REAL de C_npmi (ref_texts, sin podar por min_df)
print("\n### longitud de ref_texts — la base sobre la que corre la ventana de diez")
for ck, lab in [("ckpt_lda_multiseed.pkl","LDA"), ("ckpt_tfidf_multiseed.pkl","TF-IDF"),
                ("ckpt_w2v_multiseed.pkl","W2V"), ("ckpt_bertopic_multiseed.pkl","BERTopic")]:
    ref = pickle.load(open(f"{DFDIR}/{ck}", "rb"))['ref_texts']
    L = np.array([len(t) for t in ref]); q = np.percentile(L, [25, 50, 75])
    print(f"  {lab:<9} n={len(L):,}  total={L.sum():,}  media={L.mean():.2f}  "
          f"mediana={q[1]:.0f}  Q1={q[0]:.0f}  Q3={q[2]:.0f}  |  "
          f"<=1 token: {(L<=1).mean():.1%}  <=10 tokens: {(L<=10).mean():.1%}")

### Corpus de referencia de 19.925 y 2.000 documentos (Tablas 4.1 y 4.2)

In [ ]:
import pickle
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"

CK = [("ckpt_lda_multiseed.pkl",      "LDA"),
      ("ckpt_tfidf_multiseed.pkl",    "TF-IDF"),
      ("ckpt_w2v_multiseed.pkl",      "W2V"),
      ("ckpt_bertopic_multiseed.pkl", "BERTopic")]


def series_a_tamano(n_comun, semilla_muestreo):
    """C_npmi por semilla, con el corpus de referencia recortado a n_comun documentos.
    La submuestra es FIJA dentro de cada arquitectura (una sola extracción reutilizada
    en las diez semillas), de modo que la única variación entre semillas sigue siendo
    la inicialización del modelo y el emparejamiento del Wilcoxon se conserva."""
    rng = np.random.default_rng(semilla_muestreo)
    out = {}
    for ck, lab in CK:
        d  = pickle.load(open(f"{DFDIR}/{ck}", "rb"))
        sw = set(d['final_stopwords'])
        ref = d['ref_texts']
        sub = ref if len(ref) <= n_comun else \
              [ref[i] for i in rng.choice(len(ref), n_comun, replace=False)]
        dic = Dictionary(sub)
        vals = []
        for s in sorted(d['topic_words_raw_by_seed']):
            tp = [[w for w in row if w not in sw][:10]
                  for row in d['topic_words_raw_by_seed'][s]]
            per = CoherenceModel(topics=tp, texts=sub, dictionary=dic,
                                 coherence='c_npmi').get_coherence_per_topic()
            vals.append(float(np.nanmean(per)))
        out[lab] = np.array(vals)
        print(f"  {lab:<9} C_npmi @ n={n_comun:,}: "
              f"{np.mean(vals):+.4f} ± {np.std(vals, ddof=1):.4f}")
    return out


def holm(ps):
    """Corrección de Holm, a mano para no depender de statsmodels."""
    orden = sorted(range(len(ps)), key=lambda i: ps[i])
    out, prev = [None] * len(ps), 0.0
    for r, i in enumerate(orden):
        prev = max(prev, (len(ps) - r) * ps[i])
        out[i] = min(prev, 1.0)
    return out


# ============================================================
# BLOQUE A — Tabla 4.2 a tamaño de referencia común
# ============================================================
print("### BLOQUE A — C_npmi con el corpus de referencia igualado en 19.925 documentos")
serie = series_a_tamano(19925, 42)

print("\n### BLOQUE A — contraste pareado por semilla (Wilcoxon con signo, n = 10; Holm)")
labs  = [lab for _, lab in CK]
pares = [(a, b) for i, a in enumerate(labs) for b in labs[i + 1:]]
difs  = [float(np.median(serie[a] - serie[b])) for a, b in pares]
ps    = [float(wilcoxon(serie[a], serie[b]).pvalue) for a, b in pares]
for (a, b), d, p, ph in zip(pares, difs, ps, holm(ps)):
    print(f"  {a + ' - ' + b:<28} dif. mediana = {d:+.3f}   p = {p:.4f}   p Holm = {ph:.4f}")


# ============================================================
# BLOQUE B — la columna de n = 2.000 de la Tabla 4.1, multi-semilla
# ============================================================
print("\n### BLOQUE B — C_npmi con el corpus de referencia recortado a 2.000 documentos")
print("(reemplaza los valores de una sola semilla que hoy figuran en la Tabla 4.1)")
_ = series_a_tamano(2000, 7)


# ============================================================
# BLOQUE C — las trece reseñas sin clasificar del LLM
# ============================================================
print("\n### BLOQUE C — causas del topic_id = -1 en el enfoque LLM")
res = pd.read_csv(f"{BD}/Topic/Resultados/Topico/LLM_results.csv")
sin = res.query("topic_id == -1")
print(f"  total sin clasificar: {len(sin)}")
print(sin['reason'].value_counts().to_string())

### Distancia de cada tópico a la distribución del corpus y n efectivo del contraste en diversidad (Tabla 4.2)

In [ ]:
import pickle
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.feature_extraction.text import CountVectorizer

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"

# ============================================================
# BLOQUE A — el tópico vacuo
# AlSumait define el tópico vacuo como aquel cuya distribución sobre el vocabulario
# reproduce la distribución empírica del corpus. Es contrastable: se compara cada
# tópico contra la marginal del corpus mediante divergencia de Kullback-Leibler.
# Si T0 es el más próximo a la marginal, el informe tiene razón. Si no, el fenómeno
# es otro: masa de documentos por asignación por defecto con distribución léxica normal.
# ============================================================
print("### BLOQUE A — ¿es el T0 de LDA un tópico vacuo?\n")

from sklearn.decomposition import LatentDirichletAllocation

prep = pickle.load(open(f"{DFDIR}/prep_lemmatizado.pkl", "rb"))
cv = CountVectorizer(stop_words=prep['final_stopwords'], min_df=10, max_df=0.6, ngram_range=(1, 2))
X = cv.fit_transform(prep['corpus'])

# Re-ajuste con los parámetros EXACTOS de la corrida base (semilla 42, max_iter=100).
# Es determinista: reproduce el mismo modelo, no es una corrida nueva.
lda = LatentDirichletAllocation(n_components=6, max_iter=100, learning_method='online',
                                random_state=42, n_jobs=-1)
lda.fit(X)

phi = lda.components_ / lda.components_.sum(axis=1, keepdims=True)     # p(w | tópico)
marginal = np.asarray(X.sum(axis=0)).ravel()
marginal = marginal / marginal.sum()                                    # p(w) en el corpus

def kl(p, q, eps=1e-12):
    p = p + eps; q = q + eps
    return float(np.sum(p * np.log(p / q)))

# control de que el re-ajuste reprodujo la corrida base
df_lda = pd.read_pickle(f"{DFDIR}/df_lda_final.pkl")
coincide = (lda.transform(X).argmax(axis=1) == df_lda['tema_dominante'].to_numpy()).mean()
print(f"  control — asignaciones que coinciden con df_lda_final: {coincide:.1%}  (esperado ~100 %)")
if coincide < 0.95:
    print("  >>> El re-ajuste NO reprodujo la corrida base. Interpretar el bloque A con cautela.\n")

fn = cv.get_feature_names_out()
print(f"\n  divergencia KL de cada tópico respecto de la marginal del corpus")
print(f"  (menor = más parecido al vocabulario de fondo = más 'vacuo')\n")
filas = []
for t in range(6):
    top = ", ".join(fn[i] for i in phi[t].argsort()[::-1][:5])
    filas.append((t, kl(phi[t], marginal), top))
for t, d, top in sorted(filas, key=lambda r: r[1]):
    marca = "  <-- T0" if t == 0 else ""
    print(f"    T{t}   KL = {d:.4f}   {top}{marca}")

orden = [t for t, _, _ in sorted(filas, key=lambda r: r[1])]
print(f"\n  ranking de menor a mayor KL: {orden}")
print(f"  => T0 ocupa la posición {orden.index(0) + 1} de 6")
print("     Si es la 1.ª, el T0 se comporta como tópico vacuo y la cita de AlSumait aplica.")
print("     Si no, el fenómeno es otro y hay que decirlo con precisión.\n")

# ============================================================
# BLOQUE B — el n efectivo del Wilcoxon en TD
# ============================================================
print("\n### BLOQUE B — n efectivo del contraste BERTopic − LDA en diversidad\n")

def serie_td(ck):
    d = pickle.load(open(f"{DFDIR}/{ck}", "rb")); sw = set(d['final_stopwords'])
    out = []
    for s in sorted(d['topic_words_raw_by_seed']):
        tp = [[w for w in row if w not in sw][:10] for row in d['topic_words_raw_by_seed'][s]]
        allw = [w for t in tp for w in t]
        out.append(len(set(allw)) / len(allw))
    return np.array(out)

td_b, td_l = serie_td("ckpt_bertopic_multiseed.pkl"), serie_td("ckpt_lda_multiseed.pkl")
dif = td_b - td_l
print(f"  TD BERTopic por semilla: {np.round(td_b, 4)}")
print(f"  TD LDA      por semilla: {np.round(td_l, 4)}")
print(f"  diferencias            : {np.round(dif, 4)}")
print(f"  diferencias exactamente nulas: {(dif == 0).sum()}   ->  n efectivo = {(dif != 0).sum()}")
print(f"  mediana de la diferencia: {np.median(dif):+.4f}")
for modo in ("wilcox", "pratt", "zsplit"):
    try:
        r = wilcoxon(td_b, td_l, zero_method=modo, mode='exact' if modo == 'wilcox' else 'auto')
        print(f"  Wilcoxon zero_method='{modo}': p = {r.pvalue:.4f}   (x3 por Holm = {min(r.pvalue*3,1):.4f})")
    except Exception as e:
        print(f"  Wilcoxon zero_method='{modo}': {type(e).__name__}: {e}")
print("\n  El documento afirma que el test corre con n = 10 y distribución exacta.")
print("  Si hay diferencias nulas, el n efectivo es menor y esa afirmación hay que ajustarla.")

### Contraste con Mann-Whitney (Sección 3.5.6)

In [ ]:
# ============================================================
# ROBUSTEZ AL EMPAREJAMIENTO — contraste sobre muestras independientes
# El emparejamiento por semilla es por índice de replicación, no por una condición
# compartida. El procedimiento que corresponde a dos muestras independientes es un
# test de suma de rangos. Si sus conclusiones coinciden con las del Wilcoxon pareado,
# la conclusión no depende de la decisión de emparejar.
# Autocontenido. No reentrena nada. Menos de dos minutos.
# ============================================================
import pickle, numpy as np
from scipy.stats import wilcoxon, mannwhitneyu
from itertools import combinations
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

DFDIR = "../Topic/Dataframes"
CK = [("ckpt_lda_multiseed.pkl","LDA"), ("ckpt_tfidf_multiseed.pkl","TF-IDF"),
      ("ckpt_w2v_multiseed.pkl","W2V"), ("ckpt_bertopic_multiseed.pkl","BERTopic")]

res = {}
for ck, lab in CK:
    d = pickle.load(open(f"{DFDIR}/{ck}", "rb")); sw = set(d['final_stopwords'])
    ref = d['ref_texts']; dic = Dictionary(ref); cs, ts = [], []
    for s in sorted(d['topic_words_raw_by_seed']):
        tp = [[w for w in row if w not in sw][:10] for row in d['topic_words_raw_by_seed'][s]]
        per = CoherenceModel(topics=tp, texts=ref, dictionary=dic,
                             coherence='c_npmi').get_coherence_per_topic()
        cs.append(float(np.nanmean(per)))
        allw = [w for t in tp for w in t]; ts.append(len(set(allw))/len(allw))
    res[lab] = {'cnpmi': np.array(cs), 'td': np.array(ts)}

def holm(ps):
    o = sorted(range(len(ps)), key=lambda i: ps[i]); out, prev = [0]*len(ps), 0.0
    for r, i in enumerate(o):
        prev = max(prev, (len(ps)-r)*ps[i]); out[i] = min(prev, 1.0)
    return out

for met, nom in [('cnpmi', 'C_npmi'), ('td', 'TD')]:
    print(f"\n===== {nom} =====")
    pares = list(combinations(res.keys(), 2))
    pw = [float(wilcoxon(res[a][met], res[b][met]).pvalue) for a, b in pares]
    pm = [float(mannwhitneyu(res[a][met], res[b][met], alternative='two-sided').pvalue)
          for a, b in pares]
    hw, hm = holm(pw), holm(pm)
    print(f"  {'par':<24} {'Wilcoxon pareado':>18} {'Mann-Whitney':>16}   ¿coinciden?")
    for (a, b), x, y in zip(pares, hw, hm):
        vw = 'DIST' if x < 0.05 else 'no dist'
        vm = 'DIST' if y < 0.05 else 'no dist'
        print(f"  {a+' - '+b:<24} {x:>10.4f} {vw:>7} {y:>9.4f} {vm:>7}   "
              f"{'sí' if (x<0.05)==(y<0.05) else '>>> NO <<<'}")
print("\nSi las doce filas dicen 'sí', la conclusión no depende del emparejamiento.")